# Nutrition Recommender Development 

In [1]:
#Importing essential libraries functions from other files
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import MinMaxScaler

#Importing functions from other files
from EVALUATION_METRICS import food_catalog_coverage, intra_list_diversity, inter_list_diversity, nutrient_contribution_test

In [2]:
patientdata = pd.read_csv('clean_pcos_data.csv', low_memory=False)

# Creating a sythentic target variable: Nutrient vector
### Creating the python logic for the features to decide the level of nutrients needed
### Three Target Nutrients: Fibre, Polyunsaturated Fats, and Magnesium

In [3]:
#Nutrient reccomender python logic

def nutrient_vector_1(patient_record):

    # Set a baseline daily reccomended intake of nutrients
    magnesium = 320.0 #milligrams
    fibre = 25.0 #grams
    PUFA = 12.0 #grams

    #Fiber reccomendation logic
    #High HOMA IR or high Fasting Glucose level can indicate insulin resistance
    #From high severity to mild using if and elif
    if patient_record['HOMA_IR'] > 5 or patient_record['Fasting_Glucose_mg_dL'] > 126:
        fibre += 10

    elif 3 <= patient_record['HOMA_IR'] <= 4.9 or 112.6 <= patient_record['Fasting_Glucose_mg_dL'] <= 125:
        fibre += 7.5 

    elif 2 <= patient_record['HOMA_IR'] <= 2.9 or 100 <= patient_record['Fasting_Glucose_mg_dL'] <= 112.5:
        fibre += 5
        
    elif patient_record['HOMA_IR'] > 1.9 or patient_record['Fasting_Glucose_mg_dL'] > 99:
        fibre += 2.5

    #Polyunsatured fats (PUFAs) reccommendation logic (as a replacement for Omega 3)
    #High triglycerides and the presence of severe acne can indicate high lipids and inflammation
    #Omega 3 can lower lipid levels and combat skin inflammation
    #From high severity to mild using if and elif
    if patient_record['Triglycerides_mg_dL'] > 249 or patient_record['Acne_Severity'] == 3:
        PUFA += 8
          
    elif 200 <= patient_record['Triglycerides_mg_dL'] <= 249 or patient_record['Acne_Severity'] == 2:
        PUFA += 4

    elif 150 <= patient_record['Triglycerides_mg_dL'] <= 199 or patient_record['Acne_Severity'] == 1:
        PUFA += 2
        

    #Magnesium reccomendation logic
    #Magnesium can support insulin resistance and hormonal imbalance
    #From high severity to mild using if and elif
    if patient_record['HOMA_IR'] > 5:
        if patient_record['PCOS_Diagnosis'] == 1:
            magnesium += 80
        else:
            magnesium += 70

    elif 3 <= patient_record['HOMA_IR'] <= 4.9:
        if patient_record['PCOS_Diagnosis'] == 1:
            magnesium += 50
        else:
            magnesium += 40
        
    elif 2 <= patient_record['HOMA_IR'] <= 2.9:
        if patient_record['PCOS_Diagnosis'] == 1:
            magnesium += 35
        else:
            magnesium += 25

    #Round the nutrient figures
    return [round(fibre, 1), round(PUFA, 1), round(magnesium, 1)]


### Connecting target nutrition vector levels to suitable foods
### Cosine Similarity

In [4]:
#Target nutrients matching to nutrients names in the USDA Nutrition/Food datasets

#Loading the nutrition datasets
df_food = pd.read_csv('food.csv', low_memory=False)
df_food_nutrient = pd.read_csv('food_nutrient.csv', low_memory=False)
df_nutrient = pd.read_csv('nutrient.csv', low_memory=False)
df_category = pd.read_csv('food_category.csv', low_memory=False)

#Seperating the target nutrients using the USDA nutrient ids
usda_ids = [291, 646, 304]
df_selected_nutrients = df_nutrient[df_nutrient['nutrient_nbr'].isin(usda_ids)]

#Relational Merge Pipeline
#Linking the filtered nutrients to the bridge table
merge_part1 = pd.merge(df_food_nutrient, df_selected_nutrients, left_on='nutrient_id', right_on='id', how='inner')

#Linking the result to food name table
df_joined = pd.merge(merge_part1, df_food, on='fdc_id', how='inner')

#Converting the table from a vertical format to horizontal format
#Each row represents one unique food
food_matrix = df_joined.pivot_table(
    index='description',
    columns='name',
    values='amount',
    aggfunc='mean'
).fillna(0)

#Vector for the target nutrients 
target_nutrients = [
    'Fiber, total dietary', 
    'Fatty acids, total polyunsaturated',
    'Magnesium, Mg' 
]

filtered_food_matrix = food_matrix[target_nutrients]

#Cosine similarity reccomender function
def recommend_food(patient_vector, food_db, top_n=10):

    #Converting patient vectors to a 2D row array
    vector_array_2d = np.array(patient_vector).reshape(1, -1)

    #Calculating the Cosine Similarity across all matrix rows simultaneously 
    similar_scores = cosine_similarity(vector_array_2d, food_db)[0]

    #Compiling the results into a readable output table
    results_df = food_db.copy()
    results_df['Match Score (%)'] = np.round(similar_scores * 100, 2)

    #Sorting from highest geometric match to lowest
    return results_df.sort_values(by='Match Score (%)', ascending=False).head(top_n)


In [5]:
#Calling the food catalog coverage metric to evaluate Iteration 1
print("Evaluating Iteration 1 of Baseline Nutrient Vector and Recommender System:")
print("IF/ELIF Logic Based Vector with 3 Nutrients, Unscaled Cosine Similarity Function, Unfiltered Food Database:")
print("-----------------------------------------------------------------------------------------------------------")
baseline_coverage = food_catalog_coverage(
    patient_df=patientdata,
    food_db=filtered_food_matrix,
    vector_function=nutrient_vector_1,
    recommender_function=recommend_food,
    top_n=10
)

Evaluating Iteration 1 of Baseline Nutrient Vector and Recommender System:
IF/ELIF Logic Based Vector with 3 Nutrients, Unscaled Cosine Similarity Function, Unfiltered Food Database:
-----------------------------------------------------------------------------------------------------------
Food Recommender Catalog Coverage
Total Unique Foods in Food Database: 2254
Number of Unique Foods Recommended: 41
Percentage of Unique Foods Recommended out of Total Foods: 1.82%


### Filtered food database

In [6]:
#Seperating the target nutrients using the USDA nutrient ids
target_usda_ids = [291, 646, 304, 309, 325, 326]
df_selected_nutrients_2 = df_nutrient[df_nutrient['nutrient_nbr'].isin(target_usda_ids)]

##Fixing the filtering 
food_data = pd.read_csv('food.csv')

#Filtering for only master food records and filtering out lab tests and sub samples
valid_data_types=['foundation_food', 'sr_legacy_food']
food_data = food_data[food_data['data_type'].isin(valid_data_types)].copy()

#Grouping similar foods together
food_data['short_name'] = food_data['description'].apply(lambda x: ', '.join(str(x).split(',')[:2]))

#Relational Merge Pipeline
#Linking the filtered nutrients to the bridge table
merge_part_1 = pd.merge(df_food_nutrient, df_selected_nutrients_2, left_on='nutrient_id', right_on='id', how='inner')

#Linking the result to food name table
df_joined_3 = pd.merge(merge_part_1, food_data, on='fdc_id', how='inner')

#Using the short name to merge duplicate foods
food_matrix_2 = df_joined_3.pivot_table(
    index=['short_name', 'food_category_id'],
    columns='name',
    values='amount',
    aggfunc='mean'
).fillna(0).reset_index() #Bringing the grouped indices back as standard columns

filtered_food_matrix_2 = food_matrix_2[target_nutrients]

# New Cosine Similarity Food recommendation engine

scaler = MinMaxScaler()
food_matrix_scaled = scaler.fit_transform(filtered_food_matrix_2.values)
food_scaled = pd.DataFrame(food_matrix_scaled, columns=filtered_food_matrix_2.columns, index=filtered_food_matrix_2.index)

def food_recommender_scaled(patient_vector, food_db, food_scaled_db, scalerobj, top_n=10):
    #Applying the MinMax scaling to the patient vector before running Cosine Similarity
    #Converting the patient vector to 2D
    vector_2d = np.array(patient_vector).reshape(1, -1)

    #Scaling the patient vector using the exact same food scaler rules
    #Calculating the similarity using the scaled spaces
    scaled_patient_vector = scalerobj.transform(vector_2d)
    score_similarity = cosine_similarity(scaled_patient_vector, food_scaled_db)[0]

    #Attaching the scores back to the original unscaled food database 
    results_df = food_db.copy()
    results_df['Match Score (%):'] = np.round(score_similarity * 100, 2)

    return results_df.sort_values(by='Match Score (%):', ascending=False).head(top_n)

In [7]:
#Calling the food catalog coverage metric to evaluate Iteration 2
print("Evaluating Iteration 2 of Baseline Nutrient Vector and Recommender System:")
print("IF/ELIF Logic Based Vector with 3 Nutrients, Scaled Cosine Similarity Function, Filtered Food Database:")
print("-----------------------------------------------------------------------------------------------------------")
baseline_coverage = food_catalog_coverage(
    patient_df=patientdata,
    food_db=filtered_food_matrix_2,
    vector_function=nutrient_vector_1,
    recommender_function=food_recommender_scaled,
    food_scaled_db=food_matrix_scaled,
    scalerobj=scaler,
    top_n=10
)

Evaluating Iteration 2 of Baseline Nutrient Vector and Recommender System:
IF/ELIF Logic Based Vector with 3 Nutrients, Scaled Cosine Similarity Function, Filtered Food Database:
-----------------------------------------------------------------------------------------------------------
Food Recommender Catalog Coverage
Total Unique Foods in Food Database: 337
Number of Unique Foods Recommended: 29
Percentage of Unique Foods Recommended out of Total Foods: 8.61%


## Creating a new nutrition vector with multipliers for nutrient amounts
## 5 Nutrients: Fibre, PUFAs, Magnesium, Vitamin D, Zinc

In [8]:
#Nutrient reccomender python logic

def nutrient_vector_2(patient_record):

    # Set a baseline daily reccomended intake of nutrients
    magnesium = 320.0 #milligrams
    fibre = 25.0 #grams
    PUFA = 12.0 #grams
    zinc = 7.0 #milligrams 25mg max
    vitamin_d = 10.0 #micrograms (1000 times smaller than a milligram) max 50ug

    #Fiber reccomendation logic
    #High HOMA IR or high Fasting Glucose level can indicate insulin resistance
    #Using a continuous proportional multiplier with 15g as a safety cap      
    if patient_record['HOMA_IR'] > 1.9 or patient_record['Fasting_Glucose_mg_dL'] > 99:
        fibre_addition = patient_record['HOMA_IR'] - 1.9
        fibre += min(15.0, fibre_addition * 1.5)

    #Omega 3 / Polyunsaturated fat reccommendation logic
    #High triglycerides and the presence of severe acne can indicate high lipids and inflammation
    #Omega 3 can lower lipid levels and combat skin inflammation
    #Using a continuous proportional multiplier with 10g as a safety cap
    if 150 <= patient_record['Triglycerides_mg_dL'] > 199 and patient_record['Acne_Severity'] == 3:
        PUFA_addition = patient_record['Triglycerides_mg_dL'] - 199
        PUFA += min(10.0, PUFA_addition * 1.8)

    if 150 <= patient_record['Triglycerides_mg_dL'] > 199 and patient_record['Acne_Severity'] == 2:
        PUFA_addition = patient_record['Triglycerides_mg_dL'] - 199
        PUFA += min(10.0, PUFA_addition * 1.5)

    if 150 <= patient_record['Triglycerides_mg_dL'] > 199 and patient_record['Acne_Severity'] == 1:
        PUFA_addition = patient_record['Triglycerides_mg_dL'] - 199
        PUFA += min(10.0, PUFA_addition * 1.3)

    #Magnesium reccomendation logic
    #Magnesium can support insulin resistance and hormonal imbalance
    #Using a continuous proportional multiplier with 80mg as a safety cap
    if patient_record['HOMA_IR'] > 1.9 and patient_record['PCOS_Diagnosis'] == 1:
        magnesium_addition = patient_record['HOMA_IR'] - 1.9
        magnesium += min(80, magnesium_addition * 10)
    
    if patient_record['HOMA_IR'] > 1.9 and patient_record['PCOS_Diagnosis'] == 0:
            magnesium_addition_2 = patient_record['HOMA_IR'] - 1.9
            magnesium += min(80, magnesium_addition_2 * 7.5)
    
    #Vitamin D recommendation logic
    #Vitamin D can support patients with hormonal imbalance, insulin resistance and hirsutism
    #Using a continuous proportional multiplier with 80mg as a safety cap
    if patient_record['BMI'] > 25 and patient_record['Vitamin_D_ng_mL'] < 10:
        vitamin_d_addition = patient_record['BMI'] - 25
        vitamin_d += min(40, vitamin_d_addition * 2.5)

    if patient_record['BMI'] > 25 and patient_record['Vitamin_D_ng_mL'] < 20:
            vitamin_d_addition = patient_record['BMI'] - 25
            vitamin_d += min(40, vitamin_d_addition * 1.5)

    #Zinc recommendation logic
    #Zinc can support PCOS patients with hirsutism, alopecia
    if patient_record['Total_Testosterone_ng_dL'] > 46:
        zinc_addition = patient_record['Total_Testosterone_ng_dL'] - 46
        zinc += min(18, zinc_addition * 1.5)

    #Round the nutrient figures
    return [round(fibre, 1), round(PUFA, 1), round(magnesium, 1), round(vitamin_d, 1), round(zinc, 1)]


# Creating food matrix for 5 nutrients
### Creating the food matrix so that foods can be filtered based on the 5 specific target nutrients

In [9]:
#Target nutrients matching to nutrients names in the USDA Nutrition/Food datasets

#Calculating the total vitamin D
vit_d2 = food_matrix_2.get('Vitamin D2 (ergocalciferol)', 0)
vit_d3 = food_matrix_2.get('Vitamin D3 (cholecalciferol)', 0)
food_matrix_2['Vitamin_D_Total_UG'] = vit_d2 + vit_d3

#Vector for the target nutrients 
nutrients_5d_order = [
    'short_name',
    'food_category_id',
    'Fiber, total dietary', 
    'Fatty acids, total polyunsaturated',
    'Magnesium, Mg',
    'Vitamin_D_Total_UG',
    'Zinc, Zn'
]

#Cleaning up final column names before saving
food_matrix_5d = food_matrix_2[nutrients_5d_order].copy()
food_matrix_5d.rename(columns={'short_name': 'food_description'}, inplace=True)

#Merging with food_category.csv to get the text name of the categories
food_matrix_5d = pd.merge(
    food_matrix_5d,
    df_category[['id', 'description']],
    left_on='food_category_id',
    right_on='id',
    how='left'
)

def recommender_scaled_2(patient_vector, food_db, top_n=10):
    
    #Isolating numeric columns from the filtered database
    nutrients_5d_order = [
        'Fiber, total dietary', 
        'Fatty acids, total polyunsaturated',
        'Magnesium, Mg',
        'Vitamin_D_Total_UG',
        'Zinc, Zn'
    ]
    #Filtering
    food_numeric_matrix = food_db[nutrients_5d_order].values

    #Converting the patient vector to 2D and scaling it normally
    scaler = MinMaxScaler()
    scaled_food_db = scaler.fit_transform(food_numeric_matrix)
    vector_2d = np.array(patient_vector).reshape(1, -1)
    scaled_patient_vector = scaler.transform(vector_2d)

    #Calculating the similarity strictly across the subspace
    score_similarity = cosine_similarity(scaled_patient_vector, scaled_food_db)[0]
    
    #Creating count for the top 10 recommended foods

    #Attaching the scores back and treturning the top results
    results_df = food_db.copy()
    results_df['Match_Score (%)'] = np.round(score_similarity * 100, 2)
    return results_df.sort_values(by='Match_Score (%)', ascending=False).head(top_n)



In [10]:
#Calling the food catalog coverage metric to evaluate Iteration 2
print("Evaluating Iteration 3 of Baseline Nutrient Vector and Recommender System:")
print("Continuous Multiplier Logic Based Vector with 5 Nutrients, Scaled Cosine Similarity Function, Filtered Food Database:")
print("-----------------------------------------------------------------------------------------------------------")
baseline_coverage = food_catalog_coverage(
    patient_df=patientdata,
    food_db=food_matrix_5d,
    vector_function=nutrient_vector_2,
    recommender_function=recommender_scaled_2,
    top_n=10
)

print("-----------------------------------------------------------------------------------------------------------")
iteration_3_diversity = intra_list_diversity(
    patient_df=patientdata,
    food_db=food_matrix_5d,
    vector_function=nutrient_vector_2,
    recommender_function=recommender_scaled_2,
    top_n=10
)

print("-----------------------------------------------------------------------------------------------------------")
iteration_3_personalisation = inter_list_diversity(
    patient_df=patientdata,
    food_db=food_matrix_5d,
    vector_function=nutrient_vector_2,
    recommender_function=recommender_scaled_2,
    top_n=10
)

print("-----------------------------------------------------------------------------------------------------------")
iteration_3_personalisation = nutrient_contribution_test(
    patient_df=patientdata,
    food_db=food_matrix_5d,
    vector_function=nutrient_vector_2,
    recommender_function=recommender_scaled_2,
    top_n=10
)

Evaluating Iteration 3 of Baseline Nutrient Vector and Recommender System:
Continuous Multiplier Logic Based Vector with 5 Nutrients, Scaled Cosine Similarity Function, Filtered Food Database:
-----------------------------------------------------------------------------------------------------------
Food Recommender Catalog Coverage
Total Unique Foods in Food Database: 337
Number of Unique Foods Recommended: 57
Percentage of Unique Foods Recommended out of Total Foods: 16.91%
-----------------------------------------------------------------------------------------------------------
Recommender Intra-List Diversity
Average Diversity Score:  0.80%
-----------------------------------------------------------------------------------------------------------
System Personalisation Inter-List Diversity
Total Patient Pairs Compared: 109278
Average Personalisation Score: 59.97%
-----------------------------------------------------------------------------------------------------------
Nutrient Co

### Adding Weighted Cosine Similarity into the Food Recommender 
### To test if catalogue coverage can be improved 

In [11]:
##Weighted cosine similarity
#A dynamic clinical weight vector adds a weight to nutrients depending on a patient's specific needs

def food_recommender_weighted(patient_vector, food_db, food_scaled_db, scalerobj, top_n=10):
    #Estabilishing clinical priority weights based on the patient's specific presentation
    #Default weights are equal 
    weights = np.array([1.0, 1.0, 1.0, 1.0, 1.0])

    #If insulin resistance = severe -> prioritise Fiber (index 0) and Magnesium (index 2)
    if patient_vector[0] > 30.0 or patient_vector[2] > 380.0:
        weights[0] = 2.5 #High geometric priority to Fiber
        weights[2] = 2.0 #High geometric priority to Magnesium

    #If hyperandrogenism is severe prioritise Zinc (index 3)
    vector_2d = np.array(patient_vector).reshape(1, -1)
    scaled_patient_vector = scalerobj.transform(vector_2d)

    #Applying the clinical weights
    #Multiplying both the patient vector and database rows by weights
    weighted_patient_vector = scaled_patient_vector * weights
    weighted_food_db = food_scaled_db * weights
    
    #Calculating the similarity and returning the results
    score_similarity = cosine_similarity(weighted_patient_vector, weighted_food_db)[0]

    results_df = food_db.copy()
    results_df['Match_Score (%)'] = np.round(score_similarity * 100, 2)
    return results_df.sort_values(by='Match_Score (%)', ascending=False).head(top_n)


#Isolating numeric columns from the filtered database
#To only pass these to the scaler
nutrients_5d_order = [
    'Fiber, total dietary', 
    'Fatty acids, total polyunsaturated',
    'Magnesium, Mg',
    'Vitamin_D_Total_UG',
    'Zinc, Zn'
]

food_matrix_scaled_3 = scaler.fit_transform(food_matrix_5d[nutrients_5d_order].values)

#Rebuilding the scaled dataframe
food_scaled_2 = pd.DataFrame(
    food_matrix_scaled_3,
    columns=nutrients_5d_order,
    index=food_matrix_5d.index
)


In [17]:
#Calling the food catalog coverage metric to evaluate Iteration 2
print("Evaluating Iteration 5 of Baseline Nutrient Vector and Recommender System:")
print("Continuous Multiplier Logic Based Vector with 5 Nutrients & Weighted Cosine Similarity" \
", Scaled Cosine Similarity Function, Filtered Food Database:")
print("-----------------------------------------------------------------------------------------------------------")
baseline_coverage = food_catalog_coverage(
    patient_df=patientdata,
    food_db=food_matrix_5d,
    vector_function=nutrient_vector_2,
    recommender_function=food_recommender_weighted,
    food_scaled_db=food_scaled_2,
    scalerobj=scaler,
    top_n=10
)

print("-----------------------------------------------------------------------------------------------------------")
weighted_diversity = intra_list_diversity(
    patient_df=patientdata,
    food_db=food_matrix_5d,
    vector_function=nutrient_vector_2,
    recommender_function=food_recommender_weighted,
    food_scaled_db=food_scaled_2,
    scalerobj=scaler,
    top_n=10
)

print("-----------------------------------------------------------------------------------------------------------")
weighted_diversity = inter_list_diversity(
    patient_df=patientdata,
    food_db=food_matrix_5d,
    vector_function=nutrient_vector_2,
    recommender_function=food_recommender_weighted,
    food_scaled_db=food_scaled_2,
    scalerobj=scaler,
    top_n=10
)

print("-----------------------------------------------------------------------------------------------------------")
weighted_diversity = nutrient_contribution_test(
    patient_df=patientdata,
    food_db=food_matrix_5d,
    vector_function=nutrient_vector_2,
    recommender_function=food_recommender_weighted,
    food_scaled_db=food_scaled_2,
    scalerobj=scaler,
    top_n=10
)

Evaluating Iteration 5 of Baseline Nutrient Vector and Recommender System:
Continuous Multiplier Logic Based Vector with 5 Nutrients & Weighted Cosine Similarity, Scaled Cosine Similarity Function, Filtered Food Database:
-----------------------------------------------------------------------------------------------------------
Food Recommender Catalog Coverage
Total Unique Foods in Food Database: 337
Number of Unique Foods Recommended: 60
Percentage of Unique Foods Recommended out of Total Foods: 17.80%
-----------------------------------------------------------------------------------------------------------
Recommender Intra-List Diversity
Average Diversity Score:  0.77%
-----------------------------------------------------------------------------------------------------------
System Personalisation Inter-List Diversity
Total Patient Pairs Compared: 109278
Average Personalisation Score: 63.71%
------------------------------------------------------------------------------------------

### Diversifying the food categories recommended in the top 10

In [13]:
#Food category diversification
#Programming the recommender to return the top food matches
#while enforcing a strict limit on how many items can share the same category
food_df = pd.read_csv('food.csv')
category_df = pd.read_csv('food_category.csv')

#Renaming columns in the category description 
#To avoid conflict with the food description
clean_category_df = category_df.rename(columns={'id':'food_category_id', 'description':'category_name'})

#Merging category names onto the food list
df_food_mapped = pd.merge(
    food_df[['description', 'food_category_id']],
    clean_category_df[['food_category_id', 'category_name']],
    on = 'food_category_id',
    how='left'
).drop_duplicates(subset=['description'])

#Attaching the new category name to the food_matrix_5d data
food_matrix_5d_2 = food_matrix_5d.copy()
food_matrix_5d_2 = food_matrix_5d_2.merge(
    df_food_mapped.set_index('description')[['category_name']],
    left_index=True,
    right_index=True,
    how='left'
)

#Filling in missing categories if there are any
food_matrix_5d_2['category_name'] = food_matrix_5d_2['category_name'].fillna('Other')

#Diversified Recommender Logic
def diversified_recommendations(patient_vector, food_df, top_n=10, max_per_category=2):
    #Isolating the nutrients
    nutrients_5d_order = [
        'Fiber, total dietary', 
        'Fatty acids, total polyunsaturated',
        'Magnesium, Mg',
        'Vitamin_D_Total_UG',
        'Zinc, Zn'
    ]
    food_nutrient_matrix = food_df[nutrients_5d_order].values

    #Applying the MinMax scaling to the patient vector before running Cosine Similarity
    #Converting the patient vector to 2D
    vector_2d = np.array(patient_vector).reshape(1, -1)
    scalerobj = MinMaxScaler()
    scaled_food_db = scalerobj.fit_transform(food_nutrient_matrix)

    #Scaling the patient vector using the exact same food scaler rules
    #Calculating the similarity using the scaled spaces
    scaled_patient_vector = scalerobj.transform(vector_2d)
    score_similarity = cosine_similarity(scaled_patient_vector, scaled_food_db)[0]

    #Attaching the scores back to the original unscaled food database 
    results_df = food_df.copy()
    results_df['Match_Score'] = np.round(score_similarity * 100, 2)
    #sorting the results
    sorted_results = results_df.sort_values(by='Match_Score', ascending=False)

    #Diversification loop
    diversified_top_10 = []
    category_counts = {}

    for index, row in sorted_results.iterrows():
        #Getting the category of current food
        current_category = row['category_name']

        #Initialising the category in the tracker
        if current_category not in category_counts:
            category_counts[current_category] = 0

        #If the limit hasn't been reached for the specific category, add the food
        if category_counts[current_category] < max_per_category:
            diversified_top_10.append(row)
            category_counts[current_category] += 1
        
        #Stop the loop once 10 diverse items reached
        if len(diversified_top_10) == top_n:
            break

    #Converting the list of rows into a clean Pandas DataFrame
    return pd.DataFrame(diversified_top_10)



In [18]:
#Calling the food catalog coverage metric to evaluate Iteration 2
print("Evaluating Iteration 6 of Baseline Nutrient Vector and Recommender System:")
print("Continuous Multiplier Logic Based Vector with 5 Nutrients & Diversified Recommendations" \
", Scaled Cosine Similarity Function, Filtered Food Database:")
print("-----------------------------------------------------------------------------------------------------------")
baseline_coverage = food_catalog_coverage(
    patient_df=patientdata,
    food_db=food_matrix_5d_2,
    vector_function=nutrient_vector_2,
    recommender_function=diversified_recommendations,
    top_n=10
)

print("-----------------------------------------------------------------------------------------------------------")
diversified_list_diversity = intra_list_diversity(
    patient_df=patientdata,
    food_db=food_matrix_5d_2,
    vector_function=nutrient_vector_2,
    recommender_function=diversified_recommendations,
    top_n=10
)

print("-----------------------------------------------------------------------------------------------------------")
diversified_list_diversity = inter_list_diversity(
    patient_df=patientdata,
    food_db=food_matrix_5d_2,
    vector_function=nutrient_vector_2,
    recommender_function=diversified_recommendations,
    top_n=10
)

print("-----------------------------------------------------------------------------------------------------------")
diversified_list_diversity = nutrient_contribution_test(
    patient_df=patientdata,
    food_db=food_matrix_5d_2,
    vector_function=nutrient_vector_2,
    recommender_function=diversified_recommendations,
    top_n=10
)

Evaluating Iteration 6 of Baseline Nutrient Vector and Recommender System:
Continuous Multiplier Logic Based Vector with 5 Nutrients & Diversified Recommendations, Scaled Cosine Similarity Function, Filtered Food Database:
-----------------------------------------------------------------------------------------------------------
Food Recommender Catalog Coverage
Total Unique Foods in Food Database: 337
Number of Unique Foods Recommended: 19
Percentage of Unique Foods Recommended out of Total Foods: 5.64%
-----------------------------------------------------------------------------------------------------------
Recommender Intra-List Diversity
Average Diversity Score:  1.26%
-----------------------------------------------------------------------------------------------------------
System Personalisation Inter-List Diversity
Total Patient Pairs Compared: 109278
Average Personalisation Score: 88.80%
------------------------------------------------------------------------------------------

### Hard Clinical Constraints for all 5 Nutrients

In [15]:
#Hard clinical constraints to introduce hybrid filtering 

#Vitamin D is not present in a lot of foods
#Recommended foods show up with near zero Vitamin D due to
#Vector Dot Product
#Before running the Cosine Similarity Calculation:
#Check for a patient's elevated need for Vitamin D
#If they do, the recommender should apply a filter to the food matrix
#To restrict the search space to find items that contain Vitamin D

def recommender_hard_constraints(patient_vector, food_db, top_n=10):

    #Nutrient order
    target_fiber = patient_vector[0]
    target_pufa = patient_vector[1]
    target_magnesium = patient_vector[2]
    target_vit_d = patient_vector[3]
    target_zinc = patient_vector[4]

    #Creating copies of databases for filtering
    filtered_food_db = food_db.copy()

    #Elevated need for Magnesium
    if target_fiber > 25:
        #Filtering out the foods that don't contain a lot of Zinc
        fiber_mask = filtered_food_db['Fiber, total dietary'] > 0.5
        filtered_food_db = filtered_food_db[fiber_mask]

    #Elevated need for Magnesium
    if target_pufa > 12:
        #Filtering out the foods that don't contain a lot of Zinc
        pufa_mask = filtered_food_db['Fatty acids, total polyunsaturated'] > 0.5
        filtered_food_db = filtered_food_db[pufa_mask]

    #Elevated need for Magnesium
    if target_magnesium > 320:
        #Filtering out the foods that don't contain a lot of Zinc
        magnesium_mask = filtered_food_db['Magnesium, Mg'] > 0.5
        filtered_food_db = filtered_food_db[magnesium_mask]

    #Elevated need for vitamin D
    if target_vit_d > 10:
        #Filtering out the foods that don't contain a lot of Vitamin D
        vit_d_mask = filtered_food_db['Vitamin_D_Total_UG'] > 1.0
        filtered_food_db = filtered_food_db[vit_d_mask]

    #Elevated need for Zinc
    if target_zinc > 7:
        #Filtering out the foods that don't contain a lot of Zinc
        zinc_mask = filtered_food_db['Zinc, Zn'] > 0.5
        filtered_food_db = filtered_food_db[zinc_mask]

    #If the filtering is too restrictive and returns nothing, reset to the full database
    if filtered_food_db.empty:
        filtered_food_db = food_db.copy()
    
    #Isolating numeric columns from the filtered database
    nutrients_5d_order = [
        'Fiber, total dietary', 
        'Fatty acids, total polyunsaturated',
        'Magnesium, Mg',
        'Vitamin_D_Total_UG',
        'Zinc, Zn'
    ]
    #Filtering
    food_numeric_matrix = filtered_food_db[nutrients_5d_order].values

    #Converting the patient vector to 2D and scaling it normally
    scaler = MinMaxScaler()
    scaled_food_db = scaler.fit_transform(food_numeric_matrix)
    vector_2d = np.array(patient_vector).reshape(1, -1)
    scaled_patient_vector = scaler.transform(vector_2d)

    #Calculating the similarity strictly across the subspace
    score_similarity = cosine_similarity(scaled_patient_vector, scaled_food_db)[0]
    
    #Creating count for the top 10 recommended foods

    #Attaching the scores back and treturning the top results
    results_df = filtered_food_db.copy()
    results_df['Match_Score (%)'] = np.round(score_similarity * 100, 2)
    return results_df.sort_values(by='Match_Score (%)', ascending=False).head(top_n)

        

In [19]:
#Calling the food catalog coverage metric to evaluate Iteration 2
print("Evaluating Iteration 7 of Baseline Nutrient Vector and Recommender System:")
print("Continuous Multiplier Logic Based Vector with 5 Nutrients & Hard Clinical Constraints" \
", Scaled Cosine Similarity Function, Filtered Food Database:")
print("-----------------------------------------------------------------------------------------------------------")
baseline_coverage = food_catalog_coverage(
    patient_df=patientdata,
    food_db=food_matrix_5d,
    vector_function=nutrient_vector_2,
    recommender_function=recommender_hard_constraints,
    top_n=10
)
print("-----------------------------------------------------------------------------------------------------------")
hard_constraints_diversity = intra_list_diversity(
    patient_df=patientdata,
    food_db=food_matrix_5d,
    vector_function=nutrient_vector_2,
    recommender_function=recommender_hard_constraints,
    top_n=10
)

print("-----------------------------------------------------------------------------------------------------------")
hard_constraints_diversity = inter_list_diversity(
    patient_df=patientdata,
    food_db=food_matrix_5d,
    vector_function=nutrient_vector_2,
    recommender_function=recommender_hard_constraints,
    top_n=10
)

print("-----------------------------------------------------------------------------------------------------------")
hard_constraints_diversity = nutrient_contribution_test(
    patient_df=patientdata,
    food_db=food_matrix_5d,
    vector_function=nutrient_vector_2,
    recommender_function=recommender_hard_constraints,
    top_n=10
)

Evaluating Iteration 7 of Baseline Nutrient Vector and Recommender System:
Continuous Multiplier Logic Based Vector with 5 Nutrients & Hard Clinical Constraints, Scaled Cosine Similarity Function, Filtered Food Database:
-----------------------------------------------------------------------------------------------------------
Food Recommender Catalog Coverage
Total Unique Foods in Food Database: 337
Number of Unique Foods Recommended: 73
Percentage of Unique Foods Recommended out of Total Foods: 21.66%
-----------------------------------------------------------------------------------------------------------
Recommender Intra-List Diversity
Average Diversity Score:  1.04%
-----------------------------------------------------------------------------------------------------------
System Personalisation Inter-List Diversity
Total Patient Pairs Compared: 109278
Average Personalisation Score: 70.79%
-------------------------------------------------------------------------------------------

In [ ]:
#-------Backup nutrient vector-------
def nutrient_vector_backup(patient_record):

    #Backup nutrient vector logic for patients missing lab data
    #Relying on physical symptoms and lifestyle inputs to estimate clinical needs

    #Baseline daily recommended intake of nutrients 
     # Set a baseline daily reccomended intake of nutrients
    magnesium = 320.0 #milligrams
    fibre = 25.0 #grams
    PUFA = 12.0 #grams
    zinc = 7.0 #milligrams 25mg max
    vitamin_d = 10.0 #micrograms (1000 times smaller than a milligram) max 50ug

    #Fibre recommendation logic (to target insulin resistance proxies)
    #Acanthosis Nigricans and High BMI are strong physical indicators of IR
    if patient_record.get('Skin_Darkening_Acanthosis') == 1:
        fibre += 5.0 

    if patient_record.get('BMI') is not None and patient_record.get('BMI') > 25.0:
        fibre_addition = patient_record['BMI'] - 25.0
        fibre += min(10.0, fibre_addition * 0.5)

    #PUFA recommendation logic (To target inflammation proxies)
    #Severe acne and low physical
